# Module 10 Capstone Project Part 4

> Submitted by Phillip Amir Esguerra

In [65]:
%pip install -q langchain langchain_community langchain-pymupdf4llm langchain_huggingface sentence_transformers langchain-chroma langchain_ollama

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)



[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## Document Ingestion

The Republic Act 12305 or the Philippine National Nuclear Energy Safety Act. I got a downloadable PDF from the senate archives https://legacy.senate.gov.ph/republic_acts/ra%2012305.pdf. This was published September 18, 2025, and we should be at least sure that this does not exist in the current training data of known LLMs.

A text version can be read here: https://lawphil.net/statutes/repacts/ra2025/ra_12305_2025.html which is saved in 'ra 12305-clean.txt'

In [ ]:
from langchain_pymupdf4llm import PyMuPDF4LLMLoader

doc_path = r"ra 12305.pdf"
loader = PyMuPDF4LLMLoader(doc_path)
docs = loader.load()

In [ ]:
print(f"Loaded {len(docs)} document(s) from '{doc_path}'")

Loaded 50 document(s) from ra 12305.pdf


Create a text splitter of instance `RecursiveCharacterTextSplitter` with chunk size of 500 and 100 chunk overlap to maintain coherence with nearby chunks.

In [53]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
doc_chunks = text_splitter.split_documents(docs)

In [54]:
print(f"Chunked {len(doc_chunks)} document(s) from {doc_path}")

Chunked 269 document(s) from ra 12305.pdf


In [55]:
doc_chunks[:3]

[Document(metadata={'producer': '', 'creator': 'Image Capture Plus', 'creationdate': '2025-09-29T01:57:03+00:00', 'source': 'ra 12305.pdf', 'file_path': 'ra 12305.pdf', 'total_pages': 50, 'format': 'PDF 1.5', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2025-09-29T01:57:03+00:00', 'trapped': '', 'modDate': 'D:20250929015703Z', 'creationDate': 'D:20250929015703Z', 'page': 0}, page_content="S. No. 2899\n\nII, No. 929.1\n\n\nllcpubltc nffljepijtlippinca\n## fflmxgircsgi of pi|tltpjjttt0s\n\n\n( Mctro jillilanila\n\n\n^iucteentlj (llmigress\n\n\n®l[irb ^gular^caaion\n\n\nBegun and held in Metro Manila, on Monday, the twenty-second\nday of July, two thousand twenty-four.\n\n# 1 R epublic A ct N o . 12305]\n\n\nAN ACT PROVIDING FOR A COMPREHENSIVE LEGAL\n\nFRAMEWORK I-'OR NUCLEAR SAFETY. SECURITY.\n\nAND SAFEGUARDS IN THE PEACEFUL\n\nUTILIZATION OF NUCLEAR ENERGY IN THE\n\nPHILIPPINES. ESTABLISHING THE PHILIPPINE"),
 Document(metadata={'producer': '', 'creator': 'Ima

## Document Embedding and Vector Store Setup

We'll use [sentence-transformers/all-MiniLM-L6-v2](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2) as the embedding model and ChromaDB as the vector store.
In `HuggingFaceEmbeddings`, we store the model download cache in the same directory for later cleanup.

In [56]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model_name = "sentence-transformers/all-MiniLM-L6-v2"
embeddings = HuggingFaceEmbeddings(
    model_name=embedding_model_name, 
    cache_folder="./.cache"
)

Here, we create an instance of ChromaDB and store its data in a local directory `.chroma_db`. A collection name was also added, but we'll not use it yet.

In [58]:
from langchain_community.vectorstores import Chroma

vector_store = Chroma(
    embedding_function=embeddings,
    persist_directory="./.chroma_db",
    collection_name="ra_12305"
)

ids = vector_store.add_documents(documents=doc_chunks)

/Users/Shared/_ws/nus-gen-ai/.venv/lib/python3.12/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


In [61]:
results = vector_store.similarity_search(
    "range of civil monetary penalties"
)

print(results[0].metadata)
print("\n---\n")
print(results[0].page_content)

{'file_path': 'ra 12305.pdf', 'source': 'ra 12305.pdf', 'subject': '', 'title': '', 'modDate': 'D:20250929015703Z', 'keywords': '', 'creationDate': 'D:20250929015703Z', 'author': '', 'creator': 'Image Capture Plus', 'trapped': '', 'page': 45, 'creationdate': '2025-09-29T01:57:03+00:00', 'moddate': '2025-09-29T01:57:03+00:00', 'format': 'PDF 1.5', 'total_pages': 50, 'producer': ''}

---

Civil monetary penalties in the form of fines shall be
assessed by the PhilATOM for each violation identified which
shall not be less than Fifty thousand pesos (P50,000) but not
more than One hundred million pesos (P100,000,000);
_l_ _J_ _rouided,_ That in cases of repeated acts of non-compliance or
violation hereof, an additional fine of not less than Ten
thousand pesos (P 10,000) hut not more than One hundred
thousand pesos (P 100,000) shall be imposed for each day of
continuing violation.


/Users/Shared/_ws/nus-gen-ai/.venv/lib/python3.12/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


Visual inspecting the results, it is correct that the results were fetched from page 45 (0-based count).

## Retrieval Augmented Generation

We're going to use a local Ollama setup which we'll load llama3.1 model. The setup will use the default Ollama host at `localhost:11434`

In [75]:
# use the underlying ollama client to list all available models
from ollama import Client as OllamaClient

ollama_client = OllamaClient()
model_list = ollama_client.list()
names = [model.model for model in model_list.models]
print(names)

['embeddinggemma:latest', 'gemma3:270m', 'llama4:latest', 'phi4-reasoning:latest', 'mistral:latest', 'llama3.2:latest', 'e5-large-v2:latest', 'intfloat-e5-large-v2:latest', 'llama3.3:latest', 'llama3.1:8b-instruct-q8_0', 'phi4-mini:latest', 'qwen2.5:3b', 'jeffh/intfloat-multilingual-e5-large:f32', 'nomic-embed-text:latest', 'deepseek-r1:8b-llama-distill-q4_K_M', 'llama3.1:latest', 'deepseek-r1:latest']


In [76]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="llama3.1",
    temperature=0
)

> According to LangChain docs, `ConversationBufferMemory` is deprecated since version 0.3.1.

In [ ]:
from operator import itemgetter
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnablePassthrough
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.messages import BaseMessage
from langchain_core.chat_history import InMemoryChatMessageHistory

system_prompt = """You are a legal assistant for question-answering tasks about RA 12305 or the Philippine Nuclear Energy Safety Act.
Use the following retrieved passages to answer the question.
Cite specific sections or articles of the law when relevant.
Use concise, accurate, and formal language appropriate for legal context.
Answer in markdown format.

Context: {context}"""

history = InMemoryChatMessageHistory()

# ideally this should get the chat history from another storage i.e. DB
def get_history(thread_id: str) -> InMemoryChatMessageHistory:
    return []

retriever = vector_store.as_retriever(search_kwargs={"k": 5})
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        MessagesPlaceholder(variable_name="history"),
        ("human", "{question}"),
    ]
)
context = itemgetter("question") | retriever
rag_chain = (
    RunnablePassthrough.assign(context=context)
    | prompt
    | llm
)

chain_with_history = RunnableWithMessageHistory(
    rag_chain,
    get_history,
    input_messages_key="question",
    history_messages_key="history",
    output_messages_key="answer"
)

In [136]:
chain_with_history.invoke(
    {"question": "What is the range of civil monetary penalties for violations of the RA 12305?"}, 
    {"configurable": {"session_id": "phillip"}}
)

TypeError: 'InMemoryChatMessageHistory' object does not support item assignment

In [111]:
from IPython.display import display, Markdown, Latex

result = chain_with_history.invoke(
    "What is the range of civil monetary penalties for violations of the RA 12305?",
    config={"configurable": {"session_id": "test_session"}})

display(Markdown(result.content))

ValueError: The input to RunnablePassthrough.assign() must be a dict.

In [126]:
from operator import itemgetter
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import (
    RunnableLambda,
)
from langchain_core.runnables import RunnablePassthrough
from langchain_core.runnables.history import RunnableWithMessageHistory


def fake_retriever(query):
    assert isinstance(query, str)
    return [
        Document(page_content="人工智能快速发展"),
        Document(page_content="机器学习功不可没"),
    ]


fake_retriever = RunnableLambda(fake_retriever)

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You're an assistant who's good at something. Here is some {context}",
        ),
        MessagesPlaceholder(variable_name="history"),
        ("human", "{question}"),
    ]
)


def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


context = itemgetter("question") | fake_retriever | format_docs
first_step = RunnablePassthrough.assign(context=context)
chain = first_step | prompt | llm

with_message_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="question",
    history_messages_key="history",
)

if __name__ == '__main__':
    config = {"configurable": {"session_id": "test_session_id"}}
    result = with_message_history.invoke({"question": "请你为内容生成一个标题"}, config)
    print(result)

AttributeError: 'list' object has no attribute 'messages'